# HÂKİM — Emsalden dilekçe örnekleri (Colab)

11 milyonu **indirmez**. Sıra: `aym_bb` → `danistay` → **`yargitay` (temyiz)** → `emsal` (kısa).

`emsal` setinin başı ticaret / BAM hukuk. 400 bin satır beklemeyin; ceza yoksa 20 bin atlamadan sonra emsal **kesilir**. Yargıtay `… Ceza Dairesi` → temyiz; 25 olunca durur.

**Runtime:** Eski koşuyu **Interrupt** et. Drive’daki 45 satırı silme. Bu notebook’u yükle, **Run all**.

Log’da hemen `>>> yargitay` görmelisin; `temyiz` artmalı. `>>> emsal` + ticaret mahkemesi uzun sürerse 20k’da kesilir.

Çıktı: `MyDrive/hakim-emsal/dilekce_ornekleri_final.jsonl`

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

OUT_DIR = Path("/content/drive/MyDrive/hakim-emsal")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / "dilekce_ornekleri_final.jsonl"

REPO = "hamzabagirsakci/turkish-court-decisions"
MIN_YEAR = 2015
MIN_TEXT = 600
QUOTA = {
    "sikayet": 20,
    "suc_duyurusu": 15,
    "cevap": 15,
    "itiraz": 20,
    "istinaf": 25,
    "temyiz": 25,
    "katilma": 15,
    "bireysel_basvuru": 20,
    "idari_dava": 25,
    "tahliye": 20,
    "adli_kontrol_itiraz": 15,
}
print("çıkış", OUT_PATH)
print("hedef", sum(QUOTA.values()), QUOTA)

In [ ]:
%pip install -q datasets huggingface_hub

In [ ]:
import json
import re
from collections import Counter

from datasets import load_dataset

MADDE_RE = re.compile(
    r"\b(TCK|CMK|İYUK|IYUK|Anayasa)\s*(?:m(?:adde)?\.?\s*)?(\d{1,3})\b",
    re.I,
)
ISTISNA_RE = re.compile(
    r".{0,40}(istisna|uygulanmaz|unsurlar[ıi]\s+olu[sş]mam[ıi][sş]|"
    r"ceza verilmesine yer olmad[ıi][gğ][ıi]|hukuka ayk[ıi]r[ıi]).{0,180}",
    re.I | re.S,
)
TALEP_RE = re.compile(
    r"((?:davac[ıi]|m[uü]dafi|san[ıi]k|istinaf eden|temyiz eden|şikayet[çc]i)"
    r"[^.\n]{0,80}(?:dilekçesinde|talebinde|isteminde)[^.\n]{10,240}\.)",
    re.I,
)
OLAY_RE = re.compile(
    r"((?:OLAY VE OLGULAR|OLAYIN ÖZETİ|OLAYLAR|İDDİA)[^\n]{0,60}\n.{80,800})",
    re.I | re.S,
)

KALIP = {
    "sikayet": {
        "title": "Şikayet dilekçesi",
        "makam": "Cumhuriyet Başsavcılığı",
        "sure": "Şikayet süresi TCK m.73’e tabidir.",
        "talep": "Şikayet edilen hakkında soruşturma açılması ve kamu davası açılması talep olunur.",
    },
    "suc_duyurusu": {
        "title": "Suç duyurusu",
        "makam": "Cumhuriyet Başsavcılığı",
        "sure": "",
        "talep": "Gereğinin yapılması ve soruşturma başlatılması arz olunur.",
    },
    "cevap": {
        "title": "Cevap dilekçesi",
        "makam": "Görevli ceza mahkemesi",
        "sure": "",
        "talep": "Beraat kararı verilmesi talep olunur.",
    },
    "itiraz": {
        "title": "İtiraz dilekçesi",
        "makam": "Kararı veren merci / itiraz mercii",
        "sure": "İtiraz süresi CMK m.268 uyarınca iki hafta (14 gün) içindedir.",
        "talep": "İtirazın kabulü ile kararın kaldırılması talep olunur.",
    },
    "istinaf": {
        "title": "İstinaf dilekçesi",
        "makam": "Bölge Adliye Mahkemesi ilgili ceza dairesi",
        "sure": "İstinaf süresi CMK m.273 uyarınca tebliğden itibaren iki hafta içindedir.",
        "talep": "Hükmün kaldırılması talep olunur.",
    },
    "temyiz": {
        "title": "Temyiz dilekçesi",
        "makam": "Yargıtay ilgili ceza dairesi",
        "sure": "Temyiz süresi CMK m.291 uyarınca iki hafta (14 gün) içindedir.",
        "talep": "Kararın bozulması talep olunur.",
    },
    "katilma": {
        "title": "Davaya katılma talebi",
        "makam": "Davayı gören ceza mahkemesi",
        "sure": "",
        "talep": "CMK m.237 uyarınca davaya katılma talebinin kabulü arz olunur.",
    },
    "bireysel_basvuru": {
        "title": "Anayasa Mahkemesi bireysel başvuru",
        "makam": "Anayasa Mahkemesi",
        "sure": "Bireysel başvuru süresi 6216 sayılı Kanun m.47 uyarınca otuz gündür.",
        "talep": "İhlalin tespiti ve giderim talep olunur.",
    },
    "idari_dava": {
        "title": "İdari dava dilekçesi",
        "makam": "Görevli idare mahkemesi",
        "sure": "İdari yargıda dava açma süresi İYUK m.7 uyarınca kural olarak altmış gündür.",
        "talep": "İşlemin iptali talep olunur.",
    },
    "tahliye": {
        "title": "Tahliye talebi",
        "makam": "Tutuklamaya karar veren mahkeme / sulh ceza hakimliği",
        "sure": "",
        "talep": "Sanığın tahliyesi talep olunur.",
    },
    "adli_kontrol_itiraz": {
        "title": "Adli kontrol / tutuklama itirazı",
        "makam": "İtiraz mercii",
        "sure": "İtiraz süresi CMK m.268 uyarınca iki hafta (14 gün) içindedir.",
        "talep": "Kararın kaldırılması talep olunur.",
    },
}

CEZA_ACTIONS = {
    "sikayet", "suc_duyurusu", "cevap", "itiraz", "istinaf",
    "temyiz", "katilma", "tahliye", "adli_kontrol_itiraz",
}

JOBS = [
    ("aym_bb", ["bireysel_basvuru"]),
    ("danistay", ["idari_dava"]),
    ("yargitay", ["temyiz"]),
    ("emsal", ["istinaf"]),
]


def norm(text, n=900):
    return " ".join(str(text or "").split())[:n].strip()


def year_of(row):
    try:
        return int(row.get("year") or 0)
    except (TypeError, ValueError):
        return 0


def court_of(row):
    return str(row.get("court") or "").strip()


def court_kind(row):
    src = str(row.get("source") or "").lower()
    c = court_of(row).lower()
    if not c:
        blob = str(row.get("text") or "")[:900].lower()
        if "ağır ceza" in blob or "asliye ceza" in blob or "sulh ceza" in blob:
            return "ceza"
        return "diger"
    if src == "aym_bb" or "anayasa" in c:
        return "aym"
    if src == "danistay" or "idare mahkemesi" in c or "danıştay" in c:
        return "idare"
    if "ticaret" in c:
        return "ticaret"
    if "ceza" in c:
        return "ceza"
    if "hukuk" in c:
        return "hukuk"
    return "diger"


def classify(text, row, config=""):
    t = (text or "").lower()
    kind = court_kind(row)
    if "itiraz yoluna başvuran" in t or "itiraz konusu yasa" in t:
        return None
    if kind in {"ticaret", "hukuk"}:
        return None
    if kind == "aym":
        return "bireysel_basvuru"
    if kind == "idare":
        return "idari_dava"
    if kind != "ceza":
        return None
    if "katılma" in t or "katilma" in t:
        return "katilma"
    if "adli kontrol" in t:
        return "adli_kontrol_itiraz"
    if "tahliye" in t and "tutuklama" in t:
        return "tahliye"
    if "suç duyurusu" in t or "suc duyurusu" in t:
        return "suc_duyurusu"
    if "istinaf" in t:
        return "istinaf"
    if "temyiz" in t:
        return "temyiz"
    if "şikayet" in t or "sikayet" in t:
        return "sikayet"
    if "cevap dilekçesi" in t or "iddianameye cevap" in t:
        return "cevap"
    if "itiraz" in t and ("cmk" in t or "sulh ceza" in t or "tutuklama" in t):
        return "itiraz"
    src = str(row.get("source") or config or "").lower()
    c = court_of(row).lower()
    if (
        config == "yargitay"
        or src == "yargitay"
        or "ceza dairesi" in c
        or "ceza genel kurul" in c
    ):
        return "temyiz"
    if "bölge adliye" in c or "istinaf" in c:
        return "istinaf"
    if "sulh ceza" in c and "tutuklama" in t:
        return "adli_kontrol_itiraz"
    if "ağır ceza" in c or "asliye ceza" in c:
        return "istinaf"
    return None


def maddeler(text):
    seen, out = set(), []
    for law, no in MADDE_RE.findall(text or ""):
        key = f"{law.upper()} m.{no}"
        if key in seen:
            continue
        seen.add(key)
        out.append({"n": len(out) + 1, "madde": no, "kanun": law.upper(), "cumle": key})
        if len(out) >= 4:
            break
    return out


def istisna_span(text):
    m = ISTISNA_RE.search(text or "")
    return norm(m.group(0), 280) if m else ""


def senaryo(text):
    m = OLAY_RE.search(text or "")
    return norm(m.group(1) if m else text, 900)


def taraf_talebi(text):
    m = TALEP_RE.search(text or "")
    return norm(m.group(1), 360) if m else ""


def emsal_atif(row):
    court = court_of(row) or "ilgili mahkeme"
    esas = str(row.get("esas_no") or "").strip() or "[…]"
    karar = str(row.get("karar_no") or "").strip() or "[…]"
    if karar in {"-", "None"}:
        karar = "[…]"
    tarih = str(row.get("karar_tarihi") or "").strip() or "[…]"
    return f"{court}, {esas} esas, {karar} karar sayılı ve {tarih} tarihli ilam"


def dilekce_json(action, row, text):
    meta = KALIP[action]
    olay = senaryo(text)
    atif = emsal_atif(row)
    istisna = istisna_span(text)
    mad = maddeler(text)
    sebepler = []
    talep_ozet = taraf_talebi(text)
    if talep_ozet:
        sebepler.append(talep_ozet)
    if istisna:
        sebepler.append(f"Dayanak / istisna çerçevesi: {istisna}")
    sebepler.append(
        f"Benzer uyuşmazlıkta {atif} bu yönde değerlendirme yapmıştır; dilekçe sahibi aynı emsale dayanır."
    )
    body = {
        "makam": meta["makam"],
        "olay": olay,
        "sebepler": sebepler[:4],
        "hukuki_nitelendirme": mad,
        "emsal_atif": atif,
        "talep": meta["talep"],
    }
    if meta["sure"]:
        body["sure_cumlesi"] = meta["sure"]
    if action == "istinaf":
        body["hukum"] = "İlk derece mahkemesinin hükmü"
    elif action == "temyiz":
        body["karar"] = "Bölge adliye mahkemesi / istinaf kararı"
    elif action == "itiraz":
        body["itiraz_olunan"] = "Tebliğ edilen karar"
    elif action == "tahliye":
        body["talep_eden"] = "Sanık / müdafi"
        body["tutuklama"] = "Yürürlükteki tutuklama tedbiri"
        body["esas_no"] = str(row.get("esas_no") or "[…]")
    elif action == "sikayet":
        body["sikayetci"] = "Şikayetçi"
        body["sikayet_edilen"] = "Kimliği belirsiz şüpheli"
        body["deliller"] = ["Emsal karardaki olgu ve dayanaklar"]
        body["sure_notu"] = meta["sure"]
    elif action == "suc_duyurusu":
        body["duyuran"] = "Duyuran"
    elif action == "cevap":
        body["cevap_veren"] = "Sanık / müdafi"
        body["esas_no"] = str(row.get("esas_no") or "[…]")
        body["konu"] = "İddianameye / davaya cevap"
        body["esasa_cevap"] = olay
    elif action == "katilma":
        body["katilan"] = "Suçtan zarar gören"
        body["dava"] = "Görülmekte olan kamu davası"
        body["zarar"] = "Suçtan doğrudan zarar görüldüğü beyan olunur."
    elif action == "bireysel_basvuru":
        body["basvurucu"] = "Başvurucu"
        body["tuketilen_yollar"] = ["İstinaf", "Temyiz"]
        body["ihlal"] = "Anayasa’da güvence altına alınan hakkın ihlali iddia olunmaktadır."
    elif action == "idari_dava":
        body["davaci"] = "Davacı"
        body["davali"] = "Davalı idare"
        body["islem"] = "Dava konusu idari işlem"
    elif action == "adli_kontrol_itiraz":
        body["karar"] = "Tutuklama / adli kontrol kararı"
    return body, istisna, olay


def dilekce_metin(action, body):
    meta = KALIP[action]
    lines = [meta["makam"].upper(), "", f"Konu: {meta['title']}", "", body.get("olay", ""), ""]
    for i, s in enumerate(body.get("sebepler") or [], 1):
        lines.append(f"{i}. {s}")
    if body.get("emsal_atif"):
        lines.extend(["", f"Dayanılan emsal: {body['emsal_atif']}."])
    if body.get("sure_cumlesi"):
        lines.extend(["", body["sure_cumlesi"]])
    lines.extend(["", body.get("talep", ""), "", "Arz olunur."])
    return "\n".join(lines)


def keep(row, allowed, config=""):
    # Yargıtay akışı eski yıllardan başlar; 2015 süzgeci temyizi yok eder.
    if config != "yargitay":
        if year_of(row) and year_of(row) < MIN_YEAR:
            return None
    text = row.get("text") or row.get("karar_metni") or row.get("content") or ""
    if len(text) < MIN_TEXT:
        return None
    action = classify(text, row, config)
    if action not in allowed:
        return None
    kind = court_kind(row)
    if action == "bireysel_basvuru" and kind != "aym":
        return None
    if action == "idari_dava" and kind != "idare":
        return None
    if action in CEZA_ACTIONS and kind != "ceza":
        return None
    return action, text


print("yardımcılar hazır")

In [ ]:
seen = set()
counts = Counter()
if OUT_PATH.exists():
    for line in OUT_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        seen.add(rec["id"])
        counts[rec.get("action")] += 1
print("önceki", len(seen), dict(counts))


def quotas_full(actions):
    return all(counts[a] >= QUOTA[a] for a in actions)


def harvest(config, actions):
    left = [a for a in actions if counts[a] < QUOTA[a]]
    if not left:
        print(f"{config}: kota dolu, yapılacak bir şey yok")
        return
    print(f"\n>>> {config}  kalan { {a: QUOTA[a] - counts[a] for a in left} }")
    if config == "yargitay":
        print("Yargıtay başı hukuk; ilk Ceza Dairesi ~280 binde. O zamana kadar tutulan artmaz — normal.")
    ds = load_dataset(REPO, config, split="train", streaming=True)
    raw = 0
    kept = 0
    skipped = 0
    debug_left = 5
    give_up = 20000 if config == "emsal" else None
    with OUT_PATH.open("a", encoding="utf-8") as fh:
        for row in ds:
            if quotas_full(actions):
                print(f"{config}: kota doldu, sonraki kaynağa geçiliyor")
                break
            raw += 1
            if raw % 3000 == 0:
                print(f"  {config} satır {raw} | mahkeme {court_of(row)!r} | tutulan {dict(counts)}")
            if config in {"emsal", "yargitay"} and court_kind(row) in {"ticaret", "hukuk"}:
                skipped += 1
                if give_up and skipped >= give_up:
                    print(f"{config}: {skipped} ticaret/hukuk, ceza yok — kesildi")
                    break
                continue
            hit = keep(row, set(actions), config)
            if not hit and config == "yargitay" and court_kind(row) == "ceza" and debug_left:
                debug_left -= 1
                txt = row.get("text") or row.get("karar_metni") or ""
                print(
                    "  ceza atlandı",
                    court_of(row),
                    "yıl",
                    year_of(row),
                    "metin",
                    len(txt),
                    "sınıf",
                    classify(txt, row, config),
                    "anahtar",
                    list(row.keys())[:12],
                )
                continue
            if not hit:
                continue
            action, text = hit
            if counts[action] >= QUOTA[action]:
                continue
            rid = str(row.get("id") or f"{config}-{raw}")
            if rid in seen:
                continue
            body, istisna, olay = dilekce_json(action, row, text)
            rec = {
                "id": rid,
                "action": action,
                "senaryo": olay,
                "emsal": {
                    "source": row.get("source") or config,
                    "court": court_of(row) or None,
                    "esas_no": row.get("esas_no"),
                    "karar_no": row.get("karar_no"),
                    "karar_tarihi": row.get("karar_tarihi"),
                    "year": year_of(row) or None,
                    "atif": body.get("emsal_atif"),
                },
                "istisna": istisna,
                "dilekce": body,
                "dilekce_metin": dilekce_metin(action, body),
            }
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()
            seen.add(rid)
            counts[action] += 1
            kept += 1
    print(f"{config} bitti | satır {raw} | atlanan {skipped} | bu tur {kept} | toplam {dict(counts)}")


for config, actions in JOBS:
    harvest(config, actions)

eksik = {a: QUOTA[a] - counts[a] for a in QUOTA if counts[a] < QUOTA[a]}
print("\n=== BİTTİ ===")
print("kalıp", dict(counts))
print("toplam", sum(counts.values()), "/", sum(QUOTA.values()))
print("eksik", eksik or "yok")
print("dosya", OUT_PATH, OUT_PATH.stat().st_size if OUT_PATH.exists() else 0, "bayt")
if eksik:
    print("Eksik kaldıysa bu hücreyi tekrar çalıştır; kaldığı yerden ekler.")

In [ ]:
from collections import Counter

rows = []
if OUT_PATH.exists():
    for line in OUT_PATH.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
print("toplam", len(rows))
print("kalıp", dict(Counter(r["action"] for r in rows)))
print("kaynak", dict(Counter((r.get("emsal") or {}).get("source") for r in rows)))
print("mahkeme örneği", dict(Counter((r.get("emsal") or {}).get("court") for r in rows).most_common(8)))
if rows:
    print("\n--- ilk ---\n")
    print(rows[0]["dilekce_metin"][:900])

In [ ]:
import zipfile

zpath = OUT_DIR / "dilekce_ornekleri_final.zip"
with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(OUT_PATH, arcname="dilekce_ornekleri_final.jsonl")
print("zip", zpath)
print("PC: Drive'dan indir → data/gold/dilekce_ornekleri.jsonl üzerine koy")